In [29]:
import pandas as pd
from wikifin_rag.config import PROJECT_ROOT, DEFAULT_LLM_MODEL
from wikifin_rag.evaluation_utils import map_progress
from concurrent.futures import ThreadPoolExecutor
from uuid import uuid4
from faker import Faker
from wikifin_rag.queries import log_turn, log_feedback
import random
from datetime import datetime, timedelta

In [30]:
fake = Faker()

In [31]:
exchanges_file_path = PROJECT_ROOT / "data" / "rag-answers.csv"
sample_exchanges = pd.read_csv(exchanges_file_path)

In [32]:
sample_exchanges.head()

,doc_id,query_language,query,answer,error,reference_context
0,050dbbd0898802d4#0,nl,Hoe wordt ontvangen alimentatie belast bij de ...,Ontvangen alimentatie voor jezelf moet je aang...,NaN,"{'title': 'De ontvangen alimentatie', 'section..."
1,079bb6a60242280a#0,fr,Que puis-je faire si mon assureur demande une ...,Si votre assureur refuse une assurance solde r...,NaN,"{'title': 'Wat is de schuldsaldoverzekering?',..."
2,072662d712c69709#0,nl,Wat zijn de voordelen en regels van een oversc...,Een overschrijving heeft volgens Wikifin deze ...,NaN,"{'title': 'Wat is een overschrijving?', 'secti..."
3,077377d0e68231ee#1,nl,Hoe werkt een tegenexpertise bij een woonverze...,"Bij een woonverzekering kan je, als je het nie...",NaN,{'title': 'Woonverzekering: Wat moet je doen b...
4,02c69c65c8dfa0cc#0,en,How do I use the bingo cards for Game 4: Savin...,"For **Game 4: Savings Bingo**, the bingo cards...",NaN,"{'title': 'De Schat van Billie', 'section': 'S..."


In [33]:
SAMPLE_QUERIES = sample_exchanges["query"]
SAMPLE_ANSWERS = sample_exchanges["answer"]
RELEVANCE = ["RELEVANT", "PARTLY_RELEVANT", "IRRELEVANT"]

In [34]:
from wikifin_rag.records import ExchangeRecord, ResponseRecord, ToolCallRecord

In [35]:
def fake_exchange_record(question, answer):
    year = random.choice([2024, 2025, 2026])
    month = random.randint(1, 12)
    day = random.randint(1, 28)
    length_secs = random.randint(1, 8)

    start_dt = datetime(year=year, month=month, day=day)
    end_dt = start_dt + timedelta(seconds=length_secs)


    return ExchangeRecord(
        instructions="",
        query=question,
        model=DEFAULT_LLM_MODEL,
        response=answer,
        start_datetime=start_dt.isoformat(),
        end_datetime=end_dt.isoformat()
    )

In [36]:
def fake_response_record(exchange_record):
    start_datetime = datetime.fromisoformat(exchange_record.start_datetime)
    end_datetime = datetime.fromisoformat(exchange_record.end_datetime)
    duration_seconds = (end_datetime - start_datetime).total_seconds()
    created_offset, completed_offset = sorted(
        random.uniform(0, duration_seconds) for _ in range(2)
    )
    created_at = start_datetime + timedelta(seconds=created_offset)
    completed_at = start_datetime + timedelta(seconds=completed_offset)

    return ResponseRecord(
        id=str(uuid4()),
        exchange_id=exchange_record.id,
        created_at=created_at.isoformat(),
        completed_at=completed_at.isoformat(),
        model=exchange_record.model,
    )

In [37]:
def fake_response_records(exchange_record):
    start_datetime = datetime.fromisoformat(exchange_record.start_datetime)
    end_datetime = datetime.fromisoformat(exchange_record.end_datetime)
    response_count = random.randint(2, 4)
    duration = end_datetime - start_datetime
    response_records = []

    for index in range(response_count):
        created_at = start_datetime + duration * (index / response_count)
        completed_at = start_datetime + duration * ((index + 1) / response_count)
        response_records.append(
            ResponseRecord(
                id=str(uuid4()),
                exchange_id=exchange_record.id,
                created_at=created_at.isoformat(),
                completed_at=completed_at.isoformat(),
                model=exchange_record.model,
                input_tokens=random.randint(50, 200),
                output_tokens=random.randint(50, 300),
                input_cost=random.uniform(0.0001, 0.005),
                output_cost=random.uniform(0.0001, 0.005)
            )
        )

    return response_records

In [38]:
def fake_tool_call_record(response_record):
    return ToolCallRecord(
        id=str(uuid4()),
        call_id=str(uuid4()),
        response_id=response_record.id,
        name='search',
        arguments={'query': fake.sentence(nb_words=20, variable_nb_words=True)}
    )

In [39]:
def fake_record(question, answer):
    exchange = fake_exchange_record(question, answer)
    responses = fake_response_records(exchange)
    tool_calls = []

    for response in responses:
        if random.random() < .5:
            tool_call = fake_tool_call_record(response)
            tool_calls.append(tool_call)

    log_turn(exchange, responses, tool_calls)

    return exchange

In [40]:
def random_score():
    return random.choice([1, 1, 1, 1, 0])

In [41]:
def generate_feedback_record(exchange):
    question = exchange[0]
    answer = exchange[1]

    exchange_record = fake_record(question, answer)

    if random.random() < 0.7:
        log_feedback(
            exchange_id=exchange_record.id,
            source="judge",
            relevance=random.choice(RELEVANCE),
            explanation=""
        )

    if random.random() < 0.5:
        log_feedback(
            exchange_id=exchange_record.id,
            source="user",
            score=random_score()
        )

In [42]:
with ThreadPoolExecutor(max_workers=6) as pool:
    map_progress(pool, list(zip(SAMPLE_QUERIES, SAMPLE_ANSWERS)), generate_feedback_record)

  0%|          | 0/40 [00:00<?, ?it/s]

Error saving the turn: Error saving the turn:  [Errno 35] Resource temporarily unavailable
Error saving the turn:  [Errno 35] Resource temporarily unavailable
 [Errno 35] Resource temporarily unavailable
Error logging the user feedback for a76bb5e2-8c27-45df-b353-7cf79e1254f3: [Errno 35] Resource temporarily unavailable
Error saving the turn:  [Errno 35] Resource temporarily unavailable
